In [1]:
# Import required libraries
import logging
from typing import Any
from uuid import uuid4
import httpx

from a2a.client import A2ACardResolver, ClientFactory, ClientConfig
from a2a.types import (
    AgentCard,
    MessageSendParams,
    SendMessageRequest,
    SendStreamingMessageRequest,
)
from a2a.utils.constants import (
    AGENT_CARD_WELL_KNOWN_PATH,
    EXTENDED_AGENT_CARD_PATH,
)

# Configure logging to see what's happening
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("✓ All imports successful!")


✓ All imports successful!


In [2]:
# Configuration
base_url = 'http://localhost:10000'
timeout_seconds = 60.0

print(f"Agent Service URL: {base_url}")
print(f"Request Timeout: {timeout_seconds}s")


Agent Service URL: http://localhost:10000
Request Timeout: 60.0s


In [3]:
# Create an async HTTP client with extended timeout
httpx_client = httpx.AsyncClient(timeout=httpx.Timeout(timeout_seconds))

# Initialize the A2ACardResolver
# This helps us discover and fetch AgentCards from the service
resolver = A2ACardResolver(
    httpx_client=httpx_client,
    base_url=base_url,
)

print("✓ HTTP client and resolver initialized")


✓ HTTP client and resolver initialized


In [4]:
# Fetch the public agent card
try:
    logger.info(f'Fetching public agent card from: {base_url}{AGENT_CARD_WELL_KNOWN_PATH}')
    public_card = await resolver.get_agent_card()
    
    print("\n" + "="*60)
    print("PUBLIC AGENT CARD")
    print("="*60)
    print(public_card.model_dump_json(indent=2, exclude_none=True))
    
    # Track which card we'll use
    final_agent_card = public_card
    
    print("\n✓ Successfully fetched public agent card")
    
except Exception as e:
    logger.error(f'Failed to fetch public agent card: {e}')
    raise RuntimeError('Cannot continue without agent card') from e


INFO: Fetching public agent card from: http://localhost:10000/.well-known/agent-card.json
INFO: HTTP Request: GET http://localhost:10000/.well-known/agent-card.json "HTTP/1.1 200 OK"
INFO: Successfully fetched agent card data from http://localhost:10000/.well-known/agent-card.json: {'capabilities': {'pushNotifications': True, 'streaming': True}, 'defaultInputModes': ['text', 'text/plain'], 'defaultOutputModes': ['text', 'text/plain'], 'description': 'A helpful AI assistant with web search, academic paper search, and document retrieval capabilities', 'name': 'General Purpose Agent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': 'Search the web for current information', 'examples': ['What are the latest news about AI?'], 'id': 'web_search', 'name': 'Web Search Tool', 'tags': ['search', 'web', 'internet']}, {'description': 'Search for academic papers on arXiv', 'examples': ['Find recent papers on large language models'], 'id': 'arxiv_search', 'nam


PUBLIC AGENT CARD
{
  "capabilities": {
    "pushNotifications": true,
    "streaming": true
  },
  "defaultInputModes": [
    "text",
    "text/plain"
  ],
  "defaultOutputModes": [
    "text",
    "text/plain"
  ],
  "description": "A helpful AI assistant with web search, academic paper search, and document retrieval capabilities",
  "name": "General Purpose Agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Search the web for current information",
      "examples": [
        "What are the latest news about AI?"
      ],
      "id": "web_search",
      "name": "Web Search Tool",
      "tags": [
        "search",
        "web",
        "internet"
      ]
    },
    {
      "description": "Search for academic papers on arXiv",
      "examples": [
        "Find recent papers on large language models"
      ],
      "id": "arxiv_search",
      "name": "Academic Paper Search",
      "tags": [
        "research",
        "pap

In [5]:
# Create ClientFactory with configuration
factory = ClientFactory(
    ClientConfig(
        httpx_client=httpx_client,
        # JSON-RPC is the default transport
    )
)

# Create client using the factory and agent card
client = factory.create(card=final_agent_card)

print("✓ A2A Client initialized successfully")
print(f"  Ready to communicate with agent at: {base_url}")


✓ A2A Client initialized successfully
  Ready to communicate with agent at: http://localhost:10000


In [6]:
# Construct the message (NOT a SendMessageRequest)
message = {
    "role": "user",
    "parts": [
        {"kind": "text", "text": "What are the latest developments in artificial intelligence?"}
    ],
    "message_id": uuid4().hex,
}

print("Sending message to agent...")
print("\n" + "=" * 60)
print("STREAMING RESPONSE CHUNKS")
print("=" * 60)

chunk_count = 0
try:
    # BaseClient.send_message expects a Message (dict or model)
    # When the client is configured for streaming, this returns an async generator of events
    async for chunk in client.send_message(message):
        chunk_count += 1
        print(f"\n--- Chunk {chunk_count} ---")
        print(chunk.model_dump(mode="json", exclude_none=True))
finally:
    print("\n" + "=" * 60)
    print(f"STREAMING COMPLETE - Received {chunk_count} chunks")
    print("=" * 60)


INFO: HTTP Request: POST http://localhost:10000/ "HTTP/1.1 200 OK"


Sending message to agent...

STREAMING RESPONSE CHUNKS


INFO: New task created with id: f8e6d4da-1358-4b4b-88a3-76618e4e6472



--- Chunk 1 ---

STREAMING COMPLETE - Received 1 chunks


AttributeError: 'tuple' object has no attribute 'model_dump'

In [7]:
import json
import dataclasses
from dataclasses import asdict

# Construct the message (NOT a SendMessageRequest)
message = {
    "role": "user",
    "parts": [
        {"kind": "text", "text": "What are the latest developments in artificial intelligence?"}
    ],
    "message_id": uuid4().hex,
}

def to_jsonable(obj):
    # pydantic v2 models
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json", exclude_none=True)
    # dataclasses
    if dataclasses.is_dataclass(obj):
        return asdict(obj)
    # dicts are fine
    if isinstance(obj, dict):
        return obj
    # strings/numbers, etc.
    return obj

print("Sending message to agent...")
print("\n" + "=" * 60)
print("STREAMING RESPONSE CHUNKS")
print("=" * 60)

chunk_count = 0
try:
    # BaseClient.send_message yields streaming events
    async for chunk in client.send_message(message):
        chunk_count += 1

        # Normalize common shapes:
        # 1) (event_type, payload)
        # 2) object with .type/.delta/.message
        # 3) raw dict / model
        if isinstance(chunk, tuple) and len(chunk) == 2:
            etype, payload = chunk
        else:
            etype = getattr(chunk, "type", None)
            payload = (
                getattr(chunk, "delta", None)
                or getattr(chunk, "message", None)
                or getattr(chunk, "payload", None)
                or chunk
            )

        print(f"\n--- Chunk {chunk_count} ({etype or 'event'}) ---")

        # Prefer printing text deltas inline for a nicer UX
        if isinstance(payload, str):
            print(payload, end="", flush=True)
        else:
            print(json.dumps(to_jsonable(payload), indent=2, default=str))

finally:
    print("\n" + "=" * 60)
    print(f"STREAMING COMPLETE - Received {chunk_count} chunks")
    print("=" * 60)


INFO: HTTP Request: POST http://localhost:10000/ "HTTP/1.1 200 OK"


Sending message to agent...

STREAMING RESPONSE CHUNKS


INFO: New task created with id: 3cda359d-3f60-4655-9d30-f7b644fc60dc



--- Chunk 1 (artifacts=None context_id='b15a25de-9a86-45ba-aad1-56e49bad2686' history=[Message(context_id='b15a25de-9a86-45ba-aad1-56e49bad2686', extensions=None, kind='message', message_id='0d3837e0800241acafadb76bf36b163f', metadata=None, parts=[Part(root=TextPart(kind='text', metadata=None, text='What are the latest developments in artificial intelligence?'))], reference_task_ids=None, role=<Role.user: 'user'>, task_id='3cda359d-3f60-4655-9d30-f7b644fc60dc')] id='3cda359d-3f60-4655-9d30-f7b644fc60dc' kind='task' metadata=None status=TaskStatus(message=None, state=<TaskState.submitted: 'submitted'>, timestamp=None)) ---
null

--- Chunk 2 (artifacts=None context_id='b15a25de-9a86-45ba-aad1-56e49bad2686' history=[Message(context_id='b15a25de-9a86-45ba-aad1-56e49bad2686', extensions=None, kind='message', message_id='0d3837e0800241acafadb76bf36b163f', metadata=None, parts=[Part(root=TextPart(kind='text', metadata=None, text='What are the latest developments in artificial intelligence?')

In [11]:
# First message in a multi-turn conversation
first_message_payload = {
    'message': {
        'role': 'user',
        'parts': [
            {
                'kind': 'text',
                'text': 'Find me recent papers on transformer architectures',
            }
        ],
        'message_id': uuid4().hex,
    },
}

request = SendMessageRequest(
    id=str(uuid4()),
    params=MessageSendParams(**first_message_payload),
)

print("Sending first message in conversation...")

task_id = None
context_id = None

async for event in client.send_message(request):
    # Try to read ids from either object attrs or dict keys
    task_id = task_id or getattr(event, "id", None) or getattr(event, "task_id", None) or (isinstance(event, dict) and (event.get("id") or event.get("taskId")))
    context_id = context_id or getattr(event, "context_id", None) or (isinstance(event, dict) and event.get("contextId"))

    # Break as soon as we have both
    if task_id and context_id:
        break

print("\n" + "="*60)
print("FIRST RESPONSE (IDs)")
print("="*60)
print({"task_id": task_id, "context_id": context_id})

Sending first message in conversation...


ValidationError: 1 validation error for MessageSendParams
message
  Input should be a valid dictionary or instance of Message [type=model_type, input_value=SendMessageRequest(id='0c...d=None), metadata=None)), input_type=SendMessageRequest]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

In [12]:
# First message in a multi-turn conversation
first_message_payload = {
    'message': {
        'role': 'user',
        'parts': [{'kind': 'text', 'text': 'Find me recent papers on transformer architectures'}],
        'message_id': uuid4().hex,
    },
}

print("Sending first message in conversation...")

task_id = None
context_id = None

# Pass the message dict directly
async for event in client.send_message(first_message_payload['message']):
    task_id = task_id or getattr(event, "id", None) or getattr(event, "task_id", None) or (isinstance(event, dict) and (event.get("id") or event.get("taskId")))
    context_id = context_id or getattr(event, "context_id", None) or (isinstance(event, dict) and event.get("contextId"))
    if task_id and context_id:
        break

print("\n" + "="*60)
print("FIRST RESPONSE (IDs)")
print("="*60)
print({"task_id": task_id, "context_id": context_id})

INFO: HTTP Request: POST http://localhost:10000/ "HTTP/1.1 200 OK"


Sending first message in conversation...


INFO: New task created with id: c9522e07-478f-4f31-90ae-2eafc4d5d6eb



FIRST RESPONSE (IDs)
{'task_id': False, 'context_id': False}
